In [4]:
# operation sys + env + general
import psutil
import os
import gc
from pprint import pprint
import textwrap
from tqdm.notebook import tqdm

# utility modules
import pandas as pd
import numpy as np
import torch
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import re
import itertools

# Feature Embeddings (TF-IDF + SBERT)
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
import string

os.environ["TOKENIZERS_PARALLELISM"] = "false"
from sentence_transformers import SentenceTransformer

# LDA (Latent Dirichlet Allocation)
import gensim
import gensim.corpora as corpora
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel, Phrases
from gensim.models.phrases import Phraser

import spacy
import pyLDAvis
import pyLDAvis.gensim

# K-Means clustering
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from yellowbrick.cluster import SilhouetteVisualizer

In [5]:
import warnings
warnings.filterwarnings('ignore')

tqdm.pandas()

In [6]:
def check_memory():
    """Monitor RAM usage"""

    ram = psutil.virtual_memory()
    
    print('\n--------------')

    print(f"Memory Avaiable: {ram.available / (1024**3):.2f}GB | Memory Used: {ram.used / (1024**3):.2f}GB | Memory Free: {ram.free / (1024**3):.2f}GB | Memory Total: {ram.total/ (1024**3):.2f}GB ")
    print(f"RAM: {ram.percent}% used | ({ram.used / (1024**3):.2f}GB of {ram.total / (1024**3):.2f}GB)")

    # Display Neural Architecture Status
    print('\nArchitecture Status:')
    if torch.backends.mps.is_available():
        # MPS doesn't expose memory stats easily, but we can estimate
        print("MPS (Apple Silicon GPU) is active")
    else:
        print("MPS (Applie Silicon GPU) not active")
    
    if torch.cuda.is_available():
        print("NVIDIA CUDA is active")
    else:
        print("NVIDIA CUDA not active")
    
    print('--------------\n')


def clear_memory():
    """Aggressively clear memory between models"""

    gc.collect()

    # Clear MPS cache (for Mac)
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

    # Clear CUDA cache (for NVIDIA GPU)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print('\n--------------')
    print("Memory cleared")
    print('--------------\n')



check_memory()
clear_memory()
check_memory()


--------------
Memory Avaiable: 2.83GB | Memory Used: 5.67GB | Memory Free: 0.06GB | Memory Total: 16.00GB 
RAM: 82.3% used | (5.67GB of 16.00GB)

Architecture Status:
MPS (Apple Silicon GPU) is active
NVIDIA CUDA not active
--------------


--------------
Memory cleared
--------------


--------------
Memory Avaiable: 2.81GB | Memory Used: 5.66GB | Memory Free: 0.07GB | Memory Total: 16.00GB 
RAM: 82.4% used | (5.66GB of 16.00GB)

Architecture Status:
MPS (Apple Silicon GPU) is active
NVIDIA CUDA not active
--------------



# Load + Standardizing Data

In [7]:
reviews = pd.read_csv('../data/cleaned/combined_review_data_cleaned.csv')
reviews.drop_duplicates(inplace=True)

# NaN values in text columns to be changed to empty strings + correcting column data types
reviews['review_text'] = reviews['review_text'].fillna('')
reviews['owner_response_text'] = reviews['owner_response_text'].fillna('')

reviews = reviews.astype({
    'review_id': str,
    'source': str,
    'review_text': str,
    'owner_response_text': str
})

reviews['date_review_scraped'] = pd.to_datetime(reviews['date_review_scraped'])
reviews['review_date'] = pd.to_datetime(reviews['review_date'])

reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1589 entries, 0 to 1588
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   review_id            1589 non-null   object        
 1   rating               1589 non-null   float64       
 2   likes                1589 non-null   int64         
 3   date_review_scraped  1589 non-null   datetime64[ns]
 4   review_date          1589 non-null   datetime64[ns]
 5   company              1589 non-null   int64         
 6   source               1589 non-null   object        
 7   review_text          1589 non-null   object        
 8   owner_response_text  1589 non-null   object        
dtypes: datetime64[ns](2), float64(1), int64(2), object(4)
memory usage: 111.9+ KB


In [8]:
reviews.head(10)

,review_id,rating,likes,date_review_scraped,review_date,company,source,review_text,owner_response_text
0,ChdDSUhNMG9nS0VJQ0FnSURRaXMzbHF3RRAB,4.0,0,2026-07-26,2013-07-29,1,Google Maps,As a guy with a non complicated hair cut....th...,
1,ChdDSUhNMG9nS0VJQ0FnSURBNWJXTjdBRRAB,5.0,0,2026-07-26,2014-07-29,1,Google Maps,,
2,ChZDSUhNMG9nS0VJQ0FnSUNRb2NhQVRREAE,1.0,2,2026-07-26,2014-07-29,1,Google Maps,We have been going to Great Clips for my sons ...,
3,ChdDSUhNMG9nS0VJQ0FnSURBcXZpVnFRRRAB,5.0,0,2026-07-26,2015-07-29,1,Google Maps,Our family goes to a sweet girl named Crystal....,
4,ChdDSUhNMG9nS0VJQ0FnSUNnODhYbHFnRRAB,1.0,2,2026-07-26,2016-07-28,1,Google Maps,I show up first. Then some idiot signs in onli...,
5,ChdDSUhNMG9nS0VJQ0FnSURneTQ3SGxBRRAB,1.0,2,2026-07-26,2016-07-28,1,Google Maps,Horrible!,
6,ChdDSUhNMG9nS0VJQ0FnSUNBbnFpNnFnRRAB,5.0,0,2026-07-26,2016-07-28,1,Google Maps,I have used this location for almost 4 years. ...,
7,ChdDSUhNMG9nS0VJQ0FnSURnNmE3Rl93RRAB,3.0,0,2026-07-26,2016-07-28,1,Google Maps,,
8,ChdDSUhNMG9nS0VJQ0FnSUNBOW9XYnV3RRAB,1.0,2,2026-07-26,2016-07-28,1,Google Maps,One of worst place they don't know how to cut ...,
9,ChZDSUhNMG9nS0VJQ0FnSURBaXZhbmFREAE,1.0,2,2026-07-26,2016-07-28,1,Google Maps,I had my hair cut here yesterday. The woman wh...,


# Review Data Feature Cleaning for TF-IDF vectorizor

In [9]:
spacy_nlp_stopwords = spacy.load('en_core_web_trf').Defaults.stop_words
nltk_nlp_stopwords = set(stopwords.words('english'))

combined_stopwords = spacy_nlp_stopwords | nltk_nlp_stopwords

In [10]:
def remove_stopwords(text, stops):
    # replace all punctuation with a whitespace
    punc_to_space = str.maketrans({char: ' ' for char in string.punctuation})
    uncleaned_text = text.translate(punc_to_space)

    # remove all digits
    uncleaned_text = ''.join([i for i in uncleaned_text if not i.isdigit()])
    words = uncleaned_text.split()

    # filter out stopwords
    final = ' '.join([word for word in words if word.lower() not in stops])

    return final


def clean_docs(docs):
    final = []
    for doc in docs:
        clean_doc = remove_stopwords(doc, combined_stopwords)
        final.append(clean_doc)

    return final

In [11]:
review_docs = reviews[(reviews['review_text'] != '') & (reviews['rating'] <= 3)]['review_text'] # looking only at primarily negative reivews

review_docs_cleaned = clean_docs(review_docs)

for doc in review_docs_cleaned[:5]:
    if len(doc) > 150:
        truncated = textwrap.shorten(doc, width=100, placeholder=' ...')
        print(truncated)
    else:
        print(doc)

    print('-'*50)

going Great Clips sons husbands hair cuts years notice location going hill Yesterday straw took ...
--------------------------------------------------
idiot signs online shows min gets served time Great Clips haircut places understand come serve
--------------------------------------------------
Horrible
--------------------------------------------------
worst place know cut hair
--------------------------------------------------
hair cut yesterday woman cut terrible job catch Indian woman early pay attention asked cut ...
--------------------------------------------------


# Review Data SBERT Feature Embedding

In [12]:
sbert_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

sbert_embeddings = sbert_model.encode(np.array(review_docs))

print('Embedding shape (num_reviews, embedding_vector_size):', sbert_embeddings.shape, '\n')
print(sbert_embeddings)

Embedding shape (num_reviews, embedding_vector_size): (462, 768) 

[[ 0.02344368 -0.02551597  0.00781291 ...  0.02650463  0.01177277
   0.00984752]
 [ 0.0046633   0.00762203  0.01870975 ... -0.01355504  0.03034076
   0.00398609]
 [-0.02485966  0.03511628 -0.03872769 ... -0.02732481 -0.04573007
   0.01632192]
 ...
 [ 0.00834538  0.02715235 -0.00370851 ...  0.0078067  -0.01248601
   0.02033247]
 [-0.00180899  0.0228679   0.04249318 ...  0.03758779  0.04795991
   0.01015505]
 [-0.00182471 -0.06912671  0.00085215 ... -0.00430162 -0.03285645
  -0.02432534]]


# TF-IDF K-Means

### TF-IDF vectorizer + K-Means hyperparameter tuning and final model selection

In [13]:
def cluster_texts_with_tfidf_kmeans(
    docs,
    tfidf_param_grid=None,
    kmeans_param_grid=None,
    k_range=(2, 11),
    random_state=None,
    verbose=True,
    show_progress=True):
    """
    Hyperparameter-tune TfidfVectorizer + KMeans on text documents.

    Parameters
    ----------
    docs : list[str]
        Preprocessed documents (already passed through clean_docs()).
    tfidf_param_grid : dict or None
        Dict of lists for TfidfVectorizer params.
    kmeans_param_grid : dict or None
        Dict of lists for KMeans params excluding n_clusters.
    k_range : tuple(int, int)
        Range of k (number of clusters) to try, inclusive.
    random_state : int
        Random state for reproducibility.
    verbose : bool
        If True, print detailed status messages.
    show_progress : bool
        If True, show a tqdm progress bar in Jupyter.

    Returns
    -------
    results_df : pd.DataFrame
    best_row : pd.Series
    best_vectorizer : TfidfVectorizer
    best_kmeans : KMeans
    """

    if tfidf_param_grid is None:
        tfidf_param_grid = {
            "max_features": [500, 1000, 2000],
            "ngram_range": [(1, 1), (1, 2)],
            "min_df": [1, 2, 5],
            "max_df": [0.8, 0.95],
            "stop_words": ['english'],
            "strip_accents": ['ascii'],
            "lowercase": [True]
        }

    if kmeans_param_grid is None:
        kmeans_param_grid = {
            "init": ["k-means++"],
            "n_init": [10],
            "max_iter": [300],
        }

    tfidf_keys = list(tfidf_param_grid.keys())
    tfidf_values = list(tfidf_param_grid.values())
    tfidf_combos = [
        dict(zip(tfidf_keys, vals))
        for vals in itertools.product(*tfidf_values)
    ]

    kmeans_keys = list(kmeans_param_grid.keys())
    kmeans_values = list(kmeans_param_grid.values())
    kmeans_combos = [
        dict(zip(kmeans_keys, vals))
        for vals in itertools.product(*kmeans_values)
    ]

    k_values = list(range(k_range[0], k_range[1] + 1))
    total_iters = len(tfidf_combos) * len(kmeans_combos) * len(k_values)

    results = []

    pbar = tqdm(
        total=total_iters,
        desc="Tuning TF-IDF + KMeans",
        leave=True,
        disable=not show_progress,
    )

    for ti, tfidf_params in enumerate(tfidf_combos):
        if verbose:
            print(f"TF-IDF config {ti+1}/{len(tfidf_combos)}: {tfidf_params}")

        if show_progress:
            pbar.set_postfix({
                "tfidf_cfg": f"{ti+1}/{len(tfidf_combos)}"
            })

        vectorizer = TfidfVectorizer(**tfidf_params)
        X = vectorizer.fit_transform(docs)

        if X.shape[1] == 0:
            if verbose:
                print("  -> No features left; skipping.")

            skipped = len(kmeans_combos) * len(k_values)
            pbar.update(skipped)
            continue

        for ki, kmeans_base_params in enumerate(kmeans_combos):
            if verbose:
                print(f"  KMeans config {ki+1}/{len(kmeans_combos)}: {kmeans_base_params}")

            for k in k_values:
                kmeans_params = dict(kmeans_base_params)
                kmeans_params["n_clusters"] = k
                kmeans_params["random_state"] = random_state

                kmeans = KMeans(**kmeans_params)
                labels = kmeans.fit_predict(X)

                inertia = kmeans.inertia_

                if len(set(labels)) < 2:
                    if verbose:
                        print(f"    -> k={k} gave <2 clusters; skipping silhouette.")
                    silhouette = np.nan
                else:
                    silhouette = silhouette_score(X, labels)

                row = {
                    **tfidf_params,
                    **kmeans_params,
                    "inertia": inertia,
                    "silhouette": silhouette,
                }
                results.append(row)

                if verbose:
                    print(
                        f"    k={k:2d} | inertia={inertia:12.4f} | "
                        f"silhouette={silhouette:.4f}"
                    )

                if show_progress:
                    pbar.set_postfix({
                        "tfidf_cfg": f"{ti+1}/{len(tfidf_combos)}",
                        "k": k,
                        "silhouette": (
                            f"{silhouette:.4f}" if not np.isnan(silhouette) else "nan"
                        ),
                    })

                pbar.update(1)

    pbar.close()

    results_df = pd.DataFrame(results)

    return results_df



def construct_kmeans_tfidf(max_features, ngram_range, min_df, max_df, 
                           kmeans_init, kmeans_n_init, kmeans_max_iter, kmeans_n_clusters, kmeans_random_state):

    return_vectorizer_obj = TfidfVectorizer(
        ngram_range=ngram_range,
        max_df=max_df,
        min_df=min_df,
        max_features=max_features,
        stop_words='english',
        strip_accents='ascii',
        lowercase=True
    )
    return_kmeans_obj = KMeans(
        n_clusters=kmeans_n_clusters,
        init=kmeans_init,
        n_init=kmeans_n_init,
        max_iter=kmeans_max_iter,
        random_state=kmeans_random_state
    )

    return return_vectorizer_obj, return_kmeans_obj

In [15]:
# Hyperparamter tuning for TF-IDF matrix + K-Means

tfidf_param_grid = {
    "max_features": list(np.arange(100, 1000, 50)),
    "ngram_range": [(1,3)],
    "min_df": list(np.arange(3, 15, 1)),
    "max_df": list(round(i,2) for i in np.arange(0.25, 0.80, 0.05)),
    "stop_words": ['english'],
    "strip_accents": ['ascii'],
    "lowercase": [True]
}

kmeans_param_grid = {
    "init": ["k-means++"],
    "n_init": [10],
    "max_iter": [1000],
}

tfidf_kmeans_results_df = cluster_texts_with_tfidf_kmeans(
    review_docs_cleaned,
    tfidf_param_grid=tfidf_param_grid,
    kmeans_param_grid=kmeans_param_grid,
    k_range=(2, 20),
    random_state=100,
    verbose=False,
    show_progress=True
)

tfidf_kmeans_results_df.loc[tfidf_kmeans_results_df['n_clusters'] <= 10].sort_values('silhouette', ascending=False)

Tuning TF-IDF + KMeans:   0%|          | 0/45144 [00:00<?, ?it/s]

,max_features,ngram_range,min_df,max_df,stop_words,strip_accents,lowercase,init,n_init,max_iter,n_clusters,random_state,inertia,silhouette
2288,100,"(1, 3)",13,0.75,english,ascii,True,k-means++,10,1000,10,100,326.862883,0.068496
2231,100,"(1, 3)",13,0.60,english,ascii,True,k-means++,10,1000,10,100,326.862883,0.068496
2383,100,"(1, 3)",14,0.45,english,ascii,True,k-means++,10,1000,10,100,326.862883,0.068496
2459,100,"(1, 3)",14,0.65,english,ascii,True,k-means++,10,1000,10,100,326.862883,0.068496
2478,100,"(1, 3)",14,0.70,english,ascii,True,k-means++,10,1000,10,100,326.862883,0.068496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22751,550,"(1, 3)",3,0.70,english,ascii,True,k-means++,10,1000,10,100,395.345326,0.000059
22656,550,"(1, 3)",3,0.45,english,ascii,True,k-means++,10,1000,10,100,395.345326,0.000059
22675,550,"(1, 3)",3,0.50,english,ascii,True,k-means++,10,1000,10,100,395.345326,0.000059
22694,550,"(1, 3)",3,0.55,english,ascii,True,k-means++,10,1000,10,100,395.345326,0.000059


In [16]:
tfidf_kmeans_results_df.to_csv('../data/cleaned/tfidf_kmeans_hyperparameters_results.csv', index=False)

In [17]:
tfidf_avg_results = tfidf_kmeans_results_df.groupby('n_clusters')[['inertia', 'silhouette']].mean().reset_index()
tfidf_avg_results.sort_values('silhouette', ascending=False)

,n_clusters,inertia,silhouette
5,7,378.005625,0.036266
15,17,348.596717,0.036100
7,9,370.885424,0.035758
4,6,381.966261,0.035656
9,11,364.777307,0.035467
6,8,374.767931,0.035183
11,13,359.285770,0.035054
10,12,362.536107,0.034939
18,20,341.012258,0.034542
8,10,368.060537,0.034528


### TF-IDF K-Means Diagnostic Figures

In [18]:
# Marginalized Cluster Analysis for TF-IDF K-Means plot
fig = make_subplots(
    rows=1,
    cols=1,
    specs=[[{"secondary_y": True}]]
)

# Primary (left) y-axis: Average Inertia / WCSS
fig.add_trace(
    go.Scatter(
        x=tfidf_avg_results["n_clusters"],
        y=tfidf_avg_results["inertia"],
        mode="lines+markers",
        name="Average inertia (WCSS)",
        line=dict(
            color="#1f77b4",
            width=2
        ),
        marker=dict(
            symbol="circle",
            size=7,
            color="#1f77b4"
        ),
        hovertemplate=(
            "Number of clusters (k): %{x}<br>"
            "Average inertia (WCSS): %{y:.2f}"
            "<extra></extra>"
        )
    ),
    secondary_y=False
)

# Secondary (right) y-axis: Average Silhouette Score
fig.add_trace(
    go.Scatter(
        x=tfidf_avg_results["n_clusters"],
        y=tfidf_avg_results["silhouette"],
        mode="lines+markers",
        name="Average silhouette score",
        line=dict(
            color="maroon",
            width=2
        ),
        marker=dict(
            symbol="square",
            size=7,
            color="maroon"
        ),
        hovertemplate=(
            "Number of clusters (k): %{x}<br>"
            "Average silhouette score: %{y:.4f}"
            "<extra></extra>"
        )
    ),
    secondary_y=True
)

# Shared x-axis
fig.update_xaxes(
    title_text="Number of Clusters (k)",

    tickmode="array",
    tickvals=tfidf_avg_results["n_clusters"].tolist(),
    tickangle=0,

    showgrid=True,
    gridcolor="rgba(160, 160, 160, 0.35)",
    griddash="solid",

    showline=True,
    linecolor="rgba(80, 80, 80, 0.75)",
    linewidth=1.2,
    mirror=True,

    ticks="outside",
    ticklen=5,
    tickcolor="rgba(80, 80, 80, 0.75)",

    zeroline=False
)

# Primary / left y-axis: Inertia
fig.update_yaxes(
    title_text="Average Inertia (WCSS)",
    title_font=dict(color="#1f77b4"),
    tickfont=dict(color="#1f77b4"),

    showgrid=True,
    gridcolor="rgba(160, 160, 160, 0.35)",
    griddash="solid",

    showline=True,
    linecolor="rgba(80, 80, 80, 0.75)",
    linewidth=1.2,
    mirror=True,

    ticks="outside",
    ticklen=5,
    tickcolor="rgba(80, 80, 80, 0.75)",
    ticklabelstandoff=8,

    zeroline=False,
    secondary_y=False
)

# Secondary / right y-axis: Silhouette
fig.update_yaxes(
    title_text="Average Silhouette Score",
    title_font=dict(color="maroon"),
    tickfont=dict(color="maroon"),

    # Retain gridlines from the left axis only.
    showgrid=False,

    showline=True,
    linecolor="rgba(80, 80, 80, 0.75)",
    linewidth=1.2,

    ticks="outside",
    ticklen=5,
    tickcolor="rgba(80, 80, 80, 0.75)",
    ticklabelstandoff=8,

    zeroline=False,
    secondary_y=True
)

# Overall layout and legend
fig.update_layout(
    title=dict(
        text=(
            "<b>Marginalized Cluster Analysis for TF-IDF K-Means: "
            "Effect of K-Clusters</b>"
        ),
        x=0.5,
        xanchor="center",
        y=0.96,
        yanchor="top"
    ),
    template="plotly_white",
    width=1000,
    height=600,

    margin=dict(t=90, b=115, l=85, r=35),

    showlegend=True,
    legend=dict(
        orientation="h",
        x=0.47,
        xanchor="center",
        y=1.05,
        yanchor="middle",
        bgcolor="rgba(255,255,255,0)",
        font=dict(size=12)
    ),
    font=dict(size=14)
)

fig.show()

In [19]:
fig.write_html('../figures/TFIDF_KMeans_marginal_cluster_analysis.html')

fig.write_image('../figures/TFIDF_KMeans_marginal_cluster_analysis.png', format='png')

This plot shows that even after smoothing by taking the average of silhouette scores for all model parameters grouped by the number of clusters, the silhouette scores are highly unstable at high numbers of clusters. In addition to the low silhouette score, this indicates that this feature representation of the review data (TF-IDF matrix) and method of topic modeling (K-Means) is not nearly complex enough to capture the underlying structure of the data and as a result will highly likely produce review clustering performance that is not human interpretable

In [20]:
tfidf_kmeans_results_df.loc[tfidf_kmeans_results_df['n_clusters'] <= 7].sort_values('silhouette', ascending=False)

,max_features,ngram_range,min_df,max_df,stop_words,strip_accents,lowercase,init,n_init,max_iter,n_clusters,random_state,inertia,silhouette
2379,100,"(1, 3)",14,0.45,english,ascii,True,k-means++,10,1000,6,100,347.050212,0.060369
2246,100,"(1, 3)",13,0.65,english,ascii,True,k-means++,10,1000,6,100,347.050212,0.060369
2493,100,"(1, 3)",14,0.75,english,ascii,True,k-means++,10,1000,6,100,347.050212,0.060369
2474,100,"(1, 3)",14,0.70,english,ascii,True,k-means++,10,1000,6,100,347.050212,0.060369
2455,100,"(1, 3)",14,0.65,english,ascii,True,k-means++,10,1000,6,100,347.050212,0.060369
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40168,900,"(1, 3)",3,0.35,english,ascii,True,k-means++,10,1000,4,100,425.222344,0.007697
35172,800,"(1, 3)",3,0.40,english,ascii,True,k-means++,10,1000,5,100,416.822948,0.007548
42679,950,"(1, 3)",3,0.35,english,ascii,True,k-means++,10,1000,7,100,417.157179,0.006939
40171,900,"(1, 3)",3,0.35,english,ascii,True,k-means++,10,1000,7,100,417.157179,0.006939


In [21]:
# Cluster Analysis for TF-IDF K-Means preset
specific_config_df = tfidf_kmeans_results_df[
    (tfidf_kmeans_results_df["max_features"] == 100) &
    (tfidf_kmeans_results_df["min_df"] == 14) &
    (tfidf_kmeans_results_df["max_df"] == 0.45)
].sort_values("n_clusters")

# Make a single subplot with a secondary y-axis
fig = make_subplots(
    specs=[[{"secondary_y": True}]]
)

# Left y-axis: Inertia / WCSS
fig.add_trace(
    go.Scatter(
        x=specific_config_df["n_clusters"],
        y=specific_config_df["inertia"],
        mode="lines+markers",
        name="Inertia (WCSS)",
        line=dict(color="steelblue", width=2),
        marker=dict(
            symbol="circle",
            size=7,
            color="steelblue"
        ),
        hovertemplate=(
            "Clusters (k): %{x}<br>"
            "Inertia (WCSS): %{y:.2f}"
            "<extra></extra>"
        )
    ),
    secondary_y=False
)

# Right y-axis: Silhouette score
fig.add_trace(
    go.Scatter(
        x=specific_config_df["n_clusters"],
        y=specific_config_df["silhouette"],
        mode="lines+markers",
        name="Silhouette score",
        line=dict(color="maroon", width=2),
        marker=dict(
            symbol="square",
            size=7,
            color="maroon"
        ),
        hovertemplate=(
            "Clusters (k): %{x}<br>"
            "Silhouette score: %{y:.4f}"
            "<extra></extra>"
        )
    ),
    secondary_y=True
)

# Shared x-axis
fig.update_xaxes(
    title_text="Number of clusters (k)",
    tickmode="array",
    tickvals=specific_config_df["n_clusters"].tolist(),

    showgrid=True,
    gridcolor="rgba(160, 160, 160, 0.35)",
    griddash="solid",

    showline=True,
    linecolor="rgba(80, 80, 80, 0.75)",
    linewidth=1.2,
    mirror=True,

    ticks="outside",
    ticklen=5,
    tickcolor="rgba(80, 80, 80, 0.75)",

    zeroline=False
)

# Left y-axis styling
fig.update_yaxes(
    title_text="Inertia (WCSS)",
    title_font=dict(color="steelblue"),
    tickfont=dict(color="steelblue"),

    showgrid=True,
    gridcolor="rgba(160, 160, 160, 0.35)",
    griddash="solid",

    showline=True,
    linecolor="rgba(80, 80, 80, 0.75)",
    linewidth=1.2,
    mirror=True,

    ticks="outside",
    ticklen=5,
    tickcolor="rgba(80, 80, 80, 0.75)",

    # Increases horizontal separation between left-axis line and tick labels
    ticklabelstandoff=10,

    zeroline=False,
    secondary_y=False
)

# Right y-axis styling
fig.update_yaxes(
    title_text="Silhouette score",
    title_font=dict(color="maroon"),
    tickfont=dict(color="maroon"),

    # Keep False so only the primary y-axis renders horizontal gridlines
    showgrid=False,

    showline=True,
    linecolor="rgba(80, 80, 80, 0.75)",
    linewidth=1.2,

    ticks="outside",
    ticklen=5,
    tickcolor="rgba(80, 80, 80, 0.75)",

    # Increases horizontal separation between right-axis line and tick labels
    ticklabelstandoff=10,

    zeroline=False,
    secondary_y=True
)

# Figure title, caption, and layout
fig.update_layout(
    title=dict(
        text="<b>K-Means Clustering Analysis using TF-IDF document-term matrix</b>",
        x=0.5,
        xanchor="center",
        y=0.96,
        yanchor="top"
    ),
    template="plotly_white",
    width=1100,
    height=650,
    margin=dict(t=90, b=115, l=85, r=35),

    showlegend=True,
    legend=dict(
        orientation="h",
        x=0.47,
        xanchor="center",
        y=1.05,
        yanchor="middle",
        bgcolor="rgba(255,255,255,0)",
        font=dict(size=12)
    ),

    font=dict(size=14)
)

# Caption beneath the x-axis
fig.add_annotation(
    x=0.5,
    y=-0.18,
    xref="paper",
    yref="paper",
    text=(
        "<i>(TF-IDF vectorizer parameters: "
        "max_features=100, min_df=14, max_df=0.45)</i>"
    ),
    showarrow=False,
    xanchor="center",
    yanchor="top",
    font=dict(size=16)
)

fig.show()

In [45]:
fig.write_html('../figures/TFIDF_KMeans_cluster_anaylsis.html')

fig.write_image('../figures/TFIDF_KMeans_cluster_analysis.png', format='png')

In [22]:
# TF-IDF K-Means Diagnostic Silhouette Plots
for k in range(2, 10):
    custom_vectorizer, custom_kmeans = construct_kmeans_tfidf(
        max_features=100,
        ngram_range=(1,3),
        min_df=14,
        max_df=0.45,
        kmeans_init='k-means++',
        kmeans_n_init=10,
        kmeans_max_iter=1000,
        kmeans_n_clusters=k,
        kmeans_random_state=6740
    )

    custom_tfidf_matrix = custom_vectorizer.fit_transform(review_docs_cleaned)

    custom_silhouette_viz = SilhouetteVisualizer(custom_kmeans, 
                                                 colors='yellowbrick',
                                                 title=f'Silhouette Plot of KMeans-TFIDF Clustering (k = {k})',)
    custom_silhouette_viz.fit(custom_tfidf_matrix)

    custom_silhouette_viz.finalize()  # Prepare the figure without showing yet
    
    # Adjust font sizes
    custom_silhouette_viz.ax.set_title(
        custom_silhouette_viz.ax.get_title(), 
        fontsize=16, 
        fontweight='bold'
    )
    custom_silhouette_viz.ax.set_xlabel(
        custom_silhouette_viz.ax.get_xlabel(), 
        fontsize=14
    )
    custom_silhouette_viz.ax.set_ylabel(
        custom_silhouette_viz.ax.get_ylabel(), 
        fontsize=14
    )
    custom_silhouette_viz.ax.tick_params(labelsize=10)
    
    # Adjust legend font size if present
    if custom_silhouette_viz.ax.legend_ is not None:
        for text in custom_silhouette_viz.ax.legend_.texts:
            text.set_fontsize(11)

    plt.savefig(f'../figures/TFIDF_KMeans_silhouette_plot_k{k}.png', dpi=300, bbox_inches='tight')
    plt.close()

### TF-IDF K-Means Final Model Selection and Topic Clustering Results

In [23]:
final_vectorizer, final_kmeans = construct_kmeans_tfidf(
        max_features=100,
        ngram_range=(1,3),
        min_df=14,
        max_df=0.45,
        kmeans_init='k-means++',
        kmeans_n_init=10,
        kmeans_max_iter=1000,
        kmeans_n_clusters=6,
        kmeans_random_state=6740
    )

final_tfidf_matrix = final_vectorizer.fit_transform(review_docs_cleaned)
final_feature_names = final_vectorizer.get_feature_names_out()

final_kmeans.fit(final_tfidf_matrix)

order_features_in_centroids = final_kmeans.cluster_centers_.argsort()[:, ::-1]

width = 50  # same width as below; tweak as needed

# Header lines
header1 = 'TF-IDF Params: max_features=100, min_df=14, max_df=0.45'
header2 = 'K-Mean Params: n_clusters=3, random_state=6740'

print(header1.center(width))
print(header2.center(width))
print()                       # blank line
print(('-' * 25).center(width))
print()

for i in range(6):
    print(f"Cluster {i+1}:".center(width))
    
    top_features_idx = order_features_in_centroids[i, :25]

    for row in top_features_idx.reshape(5, 5):
        terms = [final_feature_names[id] for id in row]
        line = ', '.join(terms)
        print(line.center(width))
    
    print()

TF-IDF Params: max_features=100, min_df=14, max_df=0.45
  K-Mean Params: n_clusters=3, random_state=6740  

            -------------------------             

                    Cluster 1:                    
      hair, cut, cut hair, hair cut, stylist      
        asked, like, lady, haircut, wanted        
         short, know, want, place, going          
      guy, worst, horrible, didnt, location       
        time, uneven, bad, job, experience        

                    Cluster 2:                    
   great, clips, great clips, experience, hair    
        time, cut, went, location, stylist        
         going, got, haircut, store, left         
         said, bad, took, terrible, told          
        minutes, dont, worst, like, barber        

                    Cluster 3:                    
  haircut, worst, worst haircut, bad, experience  
      guy, stylist, location, wanted, asked       
           like, place, good, hair, ive           
          people, got,

From the result, we can see that it is true that the clustering performance is poor as the intention/topic of the cluster is not easily interpretable and contains as lot of vocabulary overlap between clusters. As mentioned previously,
this is a result of the low-complexity of this method and feature representation. More can be done such as keyword remove, which requires experimentation and more domain knowledge, to improve the results of the clustering performance,
however, it is likely that we are making the wrong assumptions about review data which will be addressed with the SBERT and LDA implementation

# SBERT K-Means

### SBERT Embedding Vector Normalization and Hyperparameter tuning

In [24]:
"""
here since the SBERT embedding are in high dimensions (768 dims), we switch to normalizing
the embedding vectors to unit-length and the dissimlarity metric is changed to cosine similarity
(1 indicates exact similar relation, 0 indicates unrelated, and -1 indicate completely opposite relation)
since this is the more classical method of comparing sentence embeddings in NLP
"""

sbert_embeddings_normalized = normalize(sbert_embeddings, norm='l2')

In [25]:
k_values = range(2, 51)
inertias = []
sil_scores = []

for k in k_values:
    km = KMeans(
        n_clusters=k,
        n_init=10,
        max_iter=1000,
        init='k-means++',
        random_state=6740
    )
    labels = km.fit_predict(sbert_embeddings_normalized)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(sbert_embeddings_normalized, labels, metric='cosine'))

sbert_kmeans_results_df = pd.DataFrame({
    "k": list(k_values),
    "inertia": inertias,
    "silhouette": sil_scores
})


sbert_kmeans_results_df.sort_values('silhouette', ascending=False).head(10)

,k,inertia,silhouette
0,2,262.372498,0.247977
2,4,241.986053,0.117424
1,3,250.122070,0.114992
3,5,234.583832,0.075442
8,10,219.479370,0.074630
4,6,230.608810,0.074496
9,11,217.334061,0.073385
15,17,207.017014,0.071563
21,23,198.632935,0.065653
29,31,189.021179,0.063120


In [26]:
sbert_kmeans_results_df.to_csv('../data/cleaned/sbert_kmeans_hyperparameter_results.csv', index=False)

### SBERT K-Means Diagnostic Figures

In [27]:
# SBERT K-Means Clustering Analysis Plot

# Sort results so the lines are drawn in increasing k order
results_plot = sbert_kmeans_results_df.sort_values("k").copy()

blue_color = "#1f77b4" 
orange_color = "#ff7f0e"
border_color = "rgba(80, 80, 80, 0.75)"
grid_color = "rgba(160, 160, 160, 0.35)"

# Create one plot with a primary and secondary y-axis
fig = make_subplots(
    rows=1,
    cols=1,
    specs=[[{"secondary_y": True}]]
)

# Primary y-axis: Inertia / WCSS
fig.add_trace(
    go.Scatter(
        x=results_plot["k"],
        y=results_plot["inertia"],
        mode="lines+markers",
        name="Inertia (WCSS)",
        line=dict(
            color=blue_color,
            width=2
        ),
        marker=dict(
            symbol="circle",
            size=7,
            color=blue_color
        ),
        hovertemplate=(
            "Clusters (k): %{x}<br>"
            "Inertia (WCSS): %{y:.2f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1,
    secondary_y=False
)

# Secondary y-axis: Silhouette score
fig.add_trace(
    go.Scatter(
        x=results_plot["k"],
        y=results_plot["silhouette"],
        mode="lines+markers",
        name="Silhouette score",
        line=dict(
            color=orange_color,
            width=2
        ),
        marker=dict(
            symbol="square",
            size=7,
            color=orange_color
        ),
        hovertemplate=(
            "Clusters (k): %{x}<br>"
            "Silhouette score: %{y:.4f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1,
    secondary_y=True
)

# X-axis: visible bottom and top borders, vertical gridlines
fig.update_xaxes(
    title_text="Number of clusters (k)",
    title_font=dict(size=13),

    tickmode="array",
    tickvals=results_plot["k"].tolist()[::3],
    tickangle=45,

    showgrid=True,
    gridcolor=grid_color,
    griddash="solid",

    showline=True,
    linecolor=border_color,
    linewidth=1.2,
    mirror=True,

    ticks="outside",
    ticklen=5,
    tickcolor=border_color,

    zeroline=False
)

# Left y-axis: Inertia / WCSS
fig.update_yaxes(
    title_text="Inertia (WCSS)",
    title_font=dict(
        size=13,
        color=blue_color
    ),
    tickfont=dict(
        color=blue_color
    ),

    showgrid=True,
    gridcolor=grid_color,
    griddash="solid",

    showline=True,
    linecolor=border_color,
    linewidth=1.2,
    mirror=True,

    ticks="outside",
    ticklen=5,
    tickcolor=border_color,

    # Horizontal gap between left axis and numeric labels
    ticklabelstandoff=10,

    zeroline=False,
    secondary_y=False
)

# Right y-axis: Silhouette score
fig.update_yaxes(
    title_text="Silhouette score",
    title_font=dict(
        size=13,
        color=orange_color
    ),
    tickfont=dict(
        color=orange_color
    ),

    # Prevents duplicate horizontal gridlines
    showgrid=False,

    showline=True,
    linecolor=border_color,
    linewidth=1.2,

    ticks="outside",
    ticklen=5,
    tickcolor=border_color,

    # Horizontal gap between right axis and numeric labels
    ticklabelstandoff=10,

    zeroline=False,
    secondary_y=True
)

# Overall figure settings
fig.update_layout(
    title=dict(
        text=(
            "<b>K-Means Clustering Analysis using "
            "L2-normalized SBERT review embeddings</b>"
        ),
        x=0.5,
        xanchor="center",
        y=0.96,
        yanchor="top",
        font=dict(size=16)
    ),

    template="plotly_white",
    width=1100,
    height=650,

    margin=dict(t=85, b=115, l=100, r=40),

    showlegend=True,
    legend=dict(
        orientation="h",
        x=0.47,
        xanchor="center",
        y=1.05,
        yanchor="middle",
        bgcolor="rgba(255,255,255,0)",
        font=dict(size=12)
    ),

    font=dict(size=14)
)

fig.show()

In [28]:
fig.write_html('../figures/SBERT_KMeans_cluster_analysis.html')

fig.write_image('../figures/SBERT_KMeans_cluster_analysis.png', format='png')

While the SBERT embeddings shows a clear improvement in the clustering analysis and a cleaner plot compared to the TF-IDF implementation at a lower number of clusters, it is still clear that silhouette scores extremely low indicating that review data points are still difficult to cleanly separate. Again, this is likely due to the underlying assumptions made about the review data. The SBERT implementation aims to improve topic clustering by using SBERT sentence transformers to map reviews into a fixed dimension vector embedding which can then be easily compared using cosine similarity rather than euclidean distance in this case. As a result, we do see marginal improvement in certain areas of the cluster analysis, particularly at a low number of clusters, suggesting that not only is the vector embeddings of the review data is potentially good direction to explore and experiment with but also that the way the review data is represented is extremely important to topic modeling performance and interpretability. In addition to this point, it is also necessary to consider not only feature representation but also assumptions made by selecting a model

In [29]:
# SBERT K-Means Diagnostic Silhouette Plots
for k in range(2,10):
    kmeans_sbert = KMeans(
        n_clusters=k,
        init='k-means++',
        n_init=10,
        max_iter=1000,
        random_state=6740
    )

    custom_kmeans_sbert_silviz = SilhouetteVisualizer(kmeans_sbert, colors='yellowbrick',
                                                      title=f"Silhouette Plot of KMeans-SBERT Clustering (k = {k})")
    custom_kmeans_sbert_silviz.fit(sbert_embeddings_normalized)

    custom_kmeans_sbert_silviz.finalize()  # Prepare the figure without showing yet
        
    # Adjust font sizes
    custom_kmeans_sbert_silviz.ax.set_title(
        custom_kmeans_sbert_silviz.ax.get_title(), 
        fontsize=16, 
        fontweight='bold'
    )
    custom_kmeans_sbert_silviz.ax.set_xlabel(
        custom_kmeans_sbert_silviz.ax.get_xlabel(), 
        fontsize=14
    )
    custom_kmeans_sbert_silviz.ax.set_ylabel(
        custom_kmeans_sbert_silviz.ax.get_ylabel(), 
        fontsize=14
    )
    custom_kmeans_sbert_silviz.ax.tick_params(labelsize=10)
    
    # Adjust legend font size if present
    if custom_kmeans_sbert_silviz.ax.legend_ is not None:
        for text in custom_kmeans_sbert_silviz.ax.legend_.texts:
            text.set_fontsize(11)


    plt.savefig(f'../figures/SBERT_KMeans_silhouette_plot_k{k}.png', dpi=300, bbox_inches='tight')
    plt.close()

### SBERT K-Means Final Model Selection + Topic Clustering Results

In [30]:
def get_top_n_representative_reviews(sbert_embeddings_normalized, review_docs, k, n):
    """
    Extract top-n most representative reviews (closest to centroid) for each cluster.
    
    Args:
        sbert_embeddings_normalized: L2-normalized SBERT embeddings (n_samples, dim)
        review_docs: pandas Series of review strings with original indices
        k: Number of clusters
        n: Number of top representative reviews
    
    Returns:
        Dictionary with top-n reviews per cluster including original indices
    """
    # Step 1: Cluster the embeddings
    km = KMeans(
        n_clusters=k,
        n_init=10,
        max_iter=1000,
        init='k-means++',
        random_state=6740
    )
    labels = km.fit_predict(sbert_embeddings_normalized)
    centroids = km.cluster_centers_
    
    # Step 2: Get original indices
    original_indices = review_docs.index.values
    
    # Step 3: Extract top 5 representatives per cluster
    cluster_top_n = {}
    
    for cluster_id in range(k):
        # Get indices of reviews in this cluster
        cluster_mask = (labels == cluster_id)
        cluster_indices_in_data = np.where(cluster_mask)[0]
        
        if len(cluster_indices_in_data) == 0:
            continue
        
        # Get embeddings for this cluster
        cluster_embeddings = sbert_embeddings_normalized[cluster_mask]
        centroid = centroids[cluster_id].reshape(1, -1)
        
        # Compute cosine similarity to centroid
        similarities = cosine_similarity(cluster_embeddings, centroid).flatten()
        
        # Get top 5 closest to centroid (or all if cluster has < 5 reviews)
        n_top = min(n, len(cluster_indices_in_data))
        top_n_idx_in_cluster = np.argsort(similarities)[-n_top:][::-1]
        
        # Store top 5 reviews with metadata
        top_n_reviews = []
        for idx_in_cluster in top_n_idx_in_cluster:
            idx_in_data = cluster_indices_in_data[idx_in_cluster]
            original_idx = original_indices[idx_in_data]
            review_text = review_docs.iloc[idx_in_data]
            
            top_n_reviews.append({
                'original_index': int(original_idx),
                'review': review_text,
                'similarity_to_centroid': float(similarities[idx_in_cluster])
            })
        
        cluster_top_n[cluster_id] = {
            'cluster_size': len(cluster_indices_in_data),
            'top_5_reviews': top_n_reviews
        }
    
    return cluster_top_n, labels


def print_cluster_summaries(cluster_top_n, wrap_width=100):
    """
    Print full text of top 5 representative reviews for each cluster with proper line wrapping.
    
    Args:
        cluster_top_5: Dictionary with top 5 reviews per cluster
        wrap_width: Maximum characters per line (default 100)
    """
    for cluster_id in sorted(cluster_top_n.keys()):
        summary = cluster_top_n[cluster_id]
        
        print(f"\n{'='*80}")
        print(f"CLUSTER {cluster_id} — {summary['cluster_size']} reviews (top-5 representative reviews)")
        print(f"{'='*80}")
        
        for i, rep in enumerate(summary['top_5_reviews'], 1):
            print(f"\n--- Representative #{i} | Cosine Similarity to Centroid: {rep['similarity_to_centroid']:.4f} ---")
            # print(f"Original Index: {rep['original_index']}")
            # print(f"Cosine Similarity to Centroid: {rep['similarity_to_centroid']:.4f}")
            print(f"\nReview:")
            
            # Wrap the review text to stay within frame
            wrapped_review = textwrap.fill(rep['review'], width=wrap_width)
            print(wrapped_review)

In [31]:
k = 2 # from hyperparameter selection of K-Means
cluster_top_n, labels = get_top_n_representative_reviews(
    sbert_embeddings_normalized, 
    review_docs, 
    k=k,
    n=5
)

print_cluster_summaries(cluster_top_n, wrap_width=75)


CLUSTER 0 — 133 reviews (top-5 representative reviews)

--- Representative #1 | Cosine Similarity to Centroid: 0.7774 ---

Review:
Rude employees, very bad experience!

--- Representative #2 | Cosine Similarity to Centroid: 0.7580 ---

Review:
Very bad service I had to walk out

--- Representative #3 | Cosine Similarity to Centroid: 0.7350 ---

Review:
Well I checked in online reached there on time yet this guy with bad sense
of humor cut other customers first who were behind in queue. He was a thin
guy of asian ethnicity not sure what his name was and he was so
unprofessional. Making stupid jokes of all kind he had hard time
pronouncing my name and called me bahubali said this is easier to call.
What a dumb thing to say. Well i liked there an elderly lady she does a
nice cut but because of this gentleman I’m not going there again.

--- Representative #4 | Cosine Similarity to Centroid: 0.7303 ---

Review:
Don't even try going to this location. Employees are rude.

--- Representative 

In both the TF-IDF and SBERT K-Means method, we implicitly assume that each review contains one topic and accordingly would be assigned to a cluster representating that topic. This represents a 'hard' clustering approach to this task of topic modeling, which is clearly not an appropriate assumption to make as reviews can contain multiple topics (e.g. customer service, quality, price, etc.). From the SBERT topic clustering results, we can see that with two topic clusters still have large topic overlap whereas the goal is to find and define clear topics. This 'hard' clustering approach made by selecting K-Means is a severe limitation in this task; an alternate approach would need to deviate from 'hard' to a more 'soft' clustering assignment where the model assumption allows for data points to belong to multiple 'topic' clusters. We can model this assumption and behavior via Gaussian Mixture Model or Latent Dirichlet Allocation, both of which allow this 'soft' clustering behavior

# Latent Dirichlet Allocation (LDA) Topic Modeling

### LDA text preprocessing

In [32]:
lda_review_docs = pd.DataFrame(review_docs)

nlp = spacy.load('en_core_web_trf')
nlp.Defaults.stop_words |= set(stopwords.words('english')) # combine spacy + nltk stop-words

In [33]:
def lda_clean_text(doc):
    text = str(doc)
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)       # keep only letters and spaces (remove digits and punctuation)
    text = re.sub(r"\s+", " ", text).strip()    # normalize whitespace (1 whitespace only)
    return text

def lemmatize_filter_text(text, token_min_len=3, allowed_postags=None, custom_remove=None):
    if pd.isna(text) or str(text).strip() == "":
        return []

    allowed_postags = set(allowed_postags) if allowed_postags is not None else None
    custom_remove = {w.lower() for w in custom_remove} if custom_remove is not None else set()

    doc = nlp(str(text).lower())
    tokens = []

    for token in doc:
        lemma = token.lemma_.lower().strip()

        if token.is_space or token.is_punct or token.is_stop: # remove stop-words
            continue
        if lemma in {"", "-pron-"}:
            continue
        if len(lemma) < token_min_len:
            continue
        if lemma in custom_remove or token in custom_remove: # remove custom-words
            continue
        if allowed_postags is not None and token.pos_ not in allowed_postags: # filter POS tags
            continue

        tokens.append(lemma)

    return tokens

def preprocess_docs(texts, token_min_len=3, allowed_postags=None, custom_remove=None):
    return [
        lemmatize_filter_text(
            text,
            token_min_len=token_min_len,
            allowed_postags=allowed_postags,
            custom_remove=custom_remove
        )
        for text in texts
    ]

def add_ngrams(token_lists, use_bigrams=True, use_trigrams=False, min_count=3, threshold=10):
    output_tokens = token_lists

    if use_bigrams:
        bigram = Phrases(output_tokens, min_count=min_count, threshold=threshold)
        bigram_mod = Phraser(bigram)
        output_tokens = [bigram_mod[doc] for doc in output_tokens]

    if use_trigrams:
        trigram = Phrases(output_tokens, min_count=min_count, threshold=threshold)
        trigram_mod = Phraser(trigram)
        output_tokens = [trigram_mod[doc] for doc in output_tokens]

    return output_tokens

def build_dictionary_corpus(token_lists, no_below=2, no_above=0.5):
    dictionary = Dictionary(token_lists)
    dictionary.filter_extremes(no_below=no_below, no_above=no_above)
    filtered_tokens = [[tok for tok in doc if tok in dictionary.token2id] for doc in token_lists]
    corpus = [dictionary.doc2bow(doc) for doc in filtered_tokens]
    return dictionary, corpus, filtered_tokens

def fit_lda_and_score(corpus, dictionary, texts, num_topics, passes=20, chunksize=50):
    model = gensim.models.ldamodel.LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=num_topics,
        random_state=100,
        update_every=1,
        chunksize=chunksize,
        passes=passes,
        alpha="auto",
        eta="auto",
        per_word_topics=True
    )

    coherence_cv = CoherenceModel(
        model=model,
        texts=texts,
        dictionary=dictionary,
        coherence="c_v"
    ).get_coherence()

    coherence_umass = CoherenceModel(
        model=model,
        texts=texts,
        dictionary=dictionary,
        coherence="u_mass"
    ).get_coherence()

    return model, coherence_cv, coherence_umass

### LDA topic and corpus-dictionary  hyperparameter tuning

In [34]:
param_grid = {
    "no_below": list(np.arange(2,11)),
    "no_above": list(np.arange(0.25, 0.85, 0.05)),
    "num_topics": list(np.arange(2, 11)),
    # "allowed_postags": [
    #     None,
    #     # ("NOUN", "ADJ"),
    #     # ("NOUN", "ADJ", "VERB")
    # ],
    # "custom_remove": [
    #     None,
    #     # ("hair", "salon"),
    #     # ("hair", "salon", "stylist")
    # ]
}

allowed_postags = None
custom_remove = None

cleaned_review_texts = lda_review_docs['review_text'].apply(lda_clean_text).tolist()

lda_results = []

token_lists = preprocess_docs(
        cleaned_review_texts,
        token_min_len=3,
        allowed_postags=allowed_postags,
        custom_remove=custom_remove
)

token_lists = add_ngrams(
        token_lists,
        use_bigrams=True,
        use_trigrams=False,
        min_count=3,
        threshold=10
    )

total_param_combinations = np.prod([len(v) for v in param_grid.values()])
param_grid_combinations = itertools.product(*param_grid.values())
param_grid_prog_bar = tqdm(param_grid_combinations, total=total_param_combinations)

for no_below, no_above, num_topics in param_grid_prog_bar:
    param_grid_prog_bar.set_description(f"no_below = {no_below} | no_above = {no_above} | num_topics = {num_topics}")

    dictionary, corpus, filtered_tokens = build_dictionary_corpus(
        token_lists,
        no_below=no_below,
        no_above=no_above
    )

    if len(dictionary) == 0:
        continue
    if all(len(doc) == 0 for doc in corpus):
        continue

    model, coherence_cv, coherence_umass = fit_lda_and_score(
        corpus=corpus,
        dictionary=dictionary,
        texts=filtered_tokens,
        num_topics=num_topics
    )

    row = {
        "no_below": no_below,
        "no_above": round(no_above, 2),
        "allowed_postags": allowed_postags,
        "custom_remove": custom_remove,
        "num_topics": num_topics,
        "coherence_cv": coherence_cv,
        "coherence_umass": coherence_umass,
        "vocab_size": len(dictionary)
    }
    lda_results.append(row) 

  0%|          | 0/972 [00:00<?, ?it/s]

In [35]:
lda_results_df = pd.DataFrame(lda_results).sort_values('coherence_umass', ascending=False)
lda_results_df[(lda_results_df['num_topics'] == 3)].head(25)

,no_below,no_above,allowed_postags,custom_remove,num_topics,coherence_cv,coherence_umass,vocab_size
1,2,0.25,None,None,3,0.573732,-2.308002,812
523,6,0.75,None,None,3,0.583772,-2.522964,289
514,6,0.70,None,None,3,0.583772,-2.522964,289
478,6,0.50,None,None,3,0.583772,-2.522964,289
496,6,0.60,None,None,3,0.583772,-2.522964,289
487,6,0.55,None,None,3,0.583772,-2.522964,289
505,6,0.65,None,None,3,0.583772,-2.522964,289
532,6,0.80,None,None,3,0.583772,-2.522964,289
577,7,0.45,None,None,3,0.613824,-2.573395,246
136,3,0.40,None,None,3,0.569514,-2.576855,580


In [36]:
lda_results_averaged = lda_results_df.groupby('num_topics')[['coherence_cv', 'coherence_umass']].mean().reset_index()
lda_results_averaged

,num_topics,coherence_cv,coherence_umass
0,2,0.594079,-2.299171
1,3,0.535312,-3.405592
2,4,0.444090,-5.678990
3,5,0.429672,-6.641627
4,6,0.417282,-7.937098
5,7,0.396804,-8.308368
6,8,0.397623,-8.728535
7,9,0.402267,-9.531147
8,10,0.398665,-9.473788


In [37]:
lda_results_df.to_csv('../data/cleaned/lda_hyperparameter_results.csv', index=False)

### LDA Diagnostic Figures

In [38]:
# Marginalized Topic Coherence Analysis
plot_df = (
    lda_results_averaged
)

# Plot styling
blue_color = "steelblue"
orange_color = "orange"
border_color = "rgba(80, 80, 80, 0.75)"
grid_color = "rgba(160, 160, 160, 0.35)"

# Create two side-by-side diagnostic plots
fig = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.12,
    subplot_titles=(
        "<b>(a) Average Topic Coherence (<i>C</i><sub>v</sub>) vs. Number of Topics</b>",
        "<b>(b) Average Topic Coherence (<i>C</i><sub>umass</sub>) vs. Number of Topics</b>"
    )
)

# (a) C_v coherence
fig.add_trace(
    go.Scatter(
        x=plot_df["num_topics"],
        y=plot_df["coherence_cv"],
        mode="lines+markers",
        name="C_v coherence",
        line=dict(
            color=blue_color,
            width=2
        ),
        marker=dict(
            symbol="circle",
            size=7,
            color=blue_color
        ),
        hovertemplate=(
            "Topics (k): %{x}<br>"
            "C<sub>v</sub>: %{y:.4f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# (b) C_umass coherence
fig.add_trace(
    go.Scatter(
        x=plot_df["num_topics"],
        y=plot_df["coherence_umass"],
        mode="lines+markers",
        name="C_umass coherence",
        line=dict(
            color=orange_color,
            width=2
        ),
        marker=dict(
            symbol="circle",
            size=7,
            color=orange_color
        ),
        hovertemplate=(
            "Topics (k): %{x}<br>"
            "C<sub>umass</sub>: %{y:.4f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# Format the left-panel x-axis
fig.update_xaxes(
    title_text="Number of Topics (<i>k</i>)",
    title_font=dict(size=14),
    tickmode="array",
    tickvals=plot_df["num_topics"].tolist(),
    tickangle=0,

    showgrid=True,
    gridcolor=grid_color,
    griddash="solid",

    showline=True,
    linecolor=border_color,
    linewidth=1.2,
    mirror=True,

    ticks="outside",
    ticklen=5,
    tickcolor=border_color,

    zeroline=False,
    row=1,
    col=1
)

# Format the right-panel x-axis
fig.update_xaxes(
    title_text="Number of Topics (<i>k</i>)",
    title_font=dict(size=14),
    tickmode="array",
    tickvals=plot_df["num_topics"].tolist(),
    tickangle=0,

    showgrid=True,
    gridcolor=grid_color,
    griddash="solid",

    showline=True,
    linecolor=border_color,
    linewidth=1.2,
    mirror=True,

    ticks="outside",
    ticklen=5,
    tickcolor=border_color,

    zeroline=False,
    row=1,
    col=2
)

# Left-panel y-axis: C_v
fig.update_yaxes(
    title_text="Average Coherence Score (<i>C</i><sub>v</sub>)",
    title_font=dict(size=14),

    showgrid=True,
    gridcolor=grid_color,
    griddash="solid",

    showline=True,
    linecolor=border_color,
    linewidth=1.2,
    mirror=True,

    ticks="outside",
    ticklen=5,
    tickcolor=border_color,
    ticklabelstandoff=8,

    zeroline=False,
    row=1,
    col=1
)

# Right-panel y-axis: C_umass
fig.update_yaxes(
    title_text="Average Coherence Score (<i>C</i><sub>umass</sub>)",
    title_font=dict(size=14),

    showgrid=True,
    gridcolor=grid_color,
    griddash="solid",

    showline=True,
    linecolor=border_color,
    linewidth=1.2,
    mirror=True,

    ticks="outside",
    ticklen=5,
    tickcolor=border_color,
    ticklabelstandoff=8,

    zeroline=False,
    row=1,
    col=2
)

# Increase the size of the two subplot titles
fig.layout.annotations[0].update(font=dict(size=16))
fig.layout.annotations[1].update(font=dict(size=16))

# Interpretive text beneath each diagnostic panel
fig.add_annotation(
    x=0.22,
    y=-0.15,
    xref="paper",
    yref="paper",
    text="<i>(C<sub>v</sub> ranges from 0 to 1; higher is better)</i>",
    showarrow=False,
    xanchor="center",
    yanchor="top",
    font=dict(size=12, color="gray")
)

fig.add_annotation(
    x=0.78,
    y=-0.15,
    xref="paper",
    yref="paper",
    text=(
        "<i>(C<sub>umass</sub> ranges from −∞ to 0; "
        "higher (less negative) is better)</i>"
    ),
    showarrow=False,
    xanchor="center",
    yanchor="top",
    font=dict(size=12, color="gray")
)

# Bottom figure caption
fig.add_annotation(
    x=0.5,
    y=-0.24,
    xref="paper",
    yref="paper",
    text=(
        "<i>(Average C<sub>v</sub> and C<sub>umass</sub> coherence scores per topic count. "
        "Values are averaged across all corpus-dictionary<br>"
        "construction and LDA algorithm settings to show the macro-trend of topic size performance)</i>"
    ),
    showarrow=False,
    xanchor="center",
    yanchor="top",
    font=dict(size=14)
)

# Overall figure layout
fig.update_layout(
    title=dict(
        text="<b>Marginalized Topic Coherence Analysis: Main Effect of Topic Count</b>",
        x=0.5,
        xanchor="center",
        y=0.97,
        yanchor="top",
        font=dict(size=20)
    ),
    template="plotly_white",
    width=1400,
    height=720,
    margin=dict(t=90, b=170, l=100,r=45),
    showlegend=False,
    font=dict(size=14)
)

fig.show()

In [39]:
fig.write_html('../figures/LDA_marginalized_topic_coherence_analysis.html')

fig.write_image('../figures/LDA_marginalized_topic_coherence_analysis.png', format='png')

Two or three topics appear to be the sweet spot in terms of number of topics for LDA. For this topic modeling task, we might choose three topics as two topics may not be granular enough to separate concerns voiced in the reviews about the business that would give us multiple operational themes to improve on.

In [40]:
# Coherence Analysis for preset Corpus-Dictionary filtering parameters
plot_df = (
    lda_results_df[
        (lda_results_df["no_below"] == 3) &
        (lda_results_df["no_above"] == 0.40)
    ]
    .sort_values("num_topics", ascending=True)
    .copy()
)

# Plot styling
blue_color = "steelblue"
orange_color = "orange"
border_color = "rgba(80, 80, 80, 0.75)"
grid_color = "rgba(160, 160, 160, 0.35)"

# Create two side-by-side diagnostic plots
fig = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.12,
    subplot_titles=(
        "<b>(a) Topic Coherence (<i>C</i><sub>v</sub>) vs. Number of Topics</b>",
        "<b>(b) Topic Coherence "
        "(<i>C</i><sub>umass</sub>) vs. Number of Topics</b>"
    )
)

# (a) C_v coherence
fig.add_trace(
    go.Scatter(
        x=plot_df["num_topics"],
        y=plot_df["coherence_cv"],
        mode="lines+markers",
        name="C_v coherence",
        line=dict(
            color=blue_color,
            width=2
        ),
        marker=dict(
            symbol="circle",
            size=7,
            color=blue_color
        ),
        hovertemplate=(
            "Topics (k): %{x}<br>"
            "C<sub>v</sub>: %{y:.4f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# (b) C_umass coherence
fig.add_trace(
    go.Scatter(
        x=plot_df["num_topics"],
        y=plot_df["coherence_umass"],
        mode="lines+markers",
        name="C_umass coherence",
        line=dict(
            color=orange_color,
            width=2
        ),
        marker=dict(
            symbol="circle",
            size=7,
            color=orange_color
        ),
        hovertemplate=(
            "Topics (k): %{x}<br>"
            "C<sub>umass</sub>: %{y:.4f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# Format the left-panel x-axis
fig.update_xaxes(
    title_text="Number of Topics (<i>k</i>)",
    title_font=dict(size=14),
    tickmode="array",
    tickvals=plot_df["num_topics"].tolist(),
    tickangle=0,

    showgrid=True,
    gridcolor=grid_color,
    griddash="solid",

    showline=True,
    linecolor=border_color,
    linewidth=1.2,
    mirror=True,

    ticks="outside",
    ticklen=5,
    tickcolor=border_color,

    zeroline=False,
    row=1,
    col=1
)

# Format the right-panel x-axis
fig.update_xaxes(
    title_text="Number of Topics (<i>k</i>)",
    title_font=dict(size=14),
    tickmode="array",
    tickvals=plot_df["num_topics"].tolist(),
    tickangle=0,

    showgrid=True,
    gridcolor=grid_color,
    griddash="solid",

    showline=True,
    linecolor=border_color,
    linewidth=1.2,
    mirror=True,

    ticks="outside",
    ticklen=5,
    tickcolor=border_color,

    zeroline=False,
    row=1,
    col=2
)

# Left-panel y-axis: C_v
fig.update_yaxes(
    title_text="Coherence Score (<i>C</i><sub>v</sub>)",
    title_font=dict(size=14),

    showgrid=True,
    gridcolor=grid_color,
    griddash="solid",

    showline=True,
    linecolor=border_color,
    linewidth=1.2,
    mirror=True,

    ticks="outside",
    ticklen=5,
    tickcolor=border_color,
    ticklabelstandoff=8,

    zeroline=False,
    row=1,
    col=1
)

# Right-panel y-axis: C_umass
fig.update_yaxes(
    title_text="Coherence Score (<i>C</i><sub>umass</sub>)",
    title_font=dict(size=14),

    showgrid=True,
    gridcolor=grid_color,
    griddash="solid",

    showline=True,
    linecolor=border_color,
    linewidth=1.2,
    mirror=True,

    ticks="outside",
    ticklen=5,
    tickcolor=border_color,
    ticklabelstandoff=8,

    zeroline=False,
    row=1,
    col=2
)

# Increase the size of the two subplot titles
fig.layout.annotations[0].update(font=dict(size=16))
fig.layout.annotations[1].update(font=dict(size=16))

# Interpretive text beneath each diagnostic panel
fig.add_annotation(
    x=0.22,
    y=-0.15,
    xref="paper",
    yref="paper",
    text="<i>(C<sub>v</sub> ranges from 0 to 1; higher is better)</i>",
    showarrow=False,
    xanchor="center",
    yanchor="top",
    font=dict(size=12, color="gray")
)

fig.add_annotation(
    x=0.78,
    y=-0.15,
    xref="paper",
    yref="paper",
    text=(
        "<i>(C<sub>umass</sub> ranges from −∞ to 0; "
        "higher (less negative) is better)</i>"
    ),
    showarrow=False,
    xanchor="center",
    yanchor="top",
    font=dict(size=12, color="gray")
)

# Bottom figure caption
fig.add_annotation(
    x=0.5,
    y=-0.24,
    xref="paper",
    yref="paper",
    text=(
        "<i>(Corpus-Dictionary filtering parameters: no_below=3, "
        "no_above=0.40)</i>"
    ),
    showarrow=False,
    xanchor="center",
    yanchor="top",
    font=dict(size=14)
)

# Overall figure layout
fig.update_layout(
    title=dict(
        text="<b>LDA Topic Coherence Analysis For Corpus-Dictionary Filtering Preset</b>",
        x=0.5,
        xanchor="center",
        y=0.97,
        yanchor="top",
        font=dict(size=20)
    ),
    template="plotly_white",
    width=1400,
    height=720,
    margin=dict(t=90, b=170, l=100,r=45),
    showlegend=False,
    font=dict(size=14)
)

fig.show()

In [41]:
fig.write_html('../figures/LDA_topic_coherence_anaylsis_CDpreset.html')

fig.write_image('../figures/LDA_topic_coherence_anaylsis_CDpreset.png', format='png')

We selected a Corpus-Dictionary filtering parameter that balances coherence scores and vocabulary size and ended up selecting `no_below = 3` and `no_above = 0.40` for the parameters which gave us a seemingly balanced vocab size of 580 (note that the vocab size ranged from ~100 to ~800 depending on filtering parameters). The coherence analysis plot for the given filtering parameter also appears to reaffirm that three topics is indeed a good choice for the topic number parameter in the LDA model as increasing the topic number from two to three does not drastically decrease $C_v$ or $C_{umass}$ scores. This helps provide a model that is more complex to better capture underlying themes

### LDA Final Model Selection + Topic Modeling Results

In [42]:
# final model parameters selected from best parameters from tuning with considerations
# of number of topics and vocabulary size
final_allowed_postags = None
final_custom_remove = None
final_no_below = 3 # 3
final_no_above = 0.40 # 0.4
final_num_topics = 3

final_token_lists = preprocess_docs(
        cleaned_review_texts,
        token_min_len=3,
        allowed_postags=final_allowed_postags,
        custom_remove=final_custom_remove
    )

final_token_lists = add_ngrams(
        final_token_lists,
        use_bigrams=True,
        use_trigrams=False,
        min_count=3,
        threshold=10
    )

final_dictionary, final_corpus, final_filtered_tokens = build_dictionary_corpus(
        final_token_lists,
        no_below=final_no_below,
        no_above=final_no_above
    )

print('Vocabulary Dictionary Size =', len(final_dictionary))

final_lda_model = gensim.models.ldamodel.LdaModel(
        corpus=final_corpus,
        id2word=final_dictionary,
        num_topics=final_num_topics,
        random_state=100,
        update_every=1,
        chunksize=50,
        passes=20,
        alpha="auto",
        eta="auto",
        per_word_topics=True
    )


print('C_v =', CoherenceModel(
        model=final_lda_model,
        texts=final_filtered_tokens,
        dictionary=final_dictionary,
        coherence="c_v"
    ).get_coherence())

print('C_umass =', CoherenceModel(
        model=final_lda_model,
        texts=final_filtered_tokens,
        dictionary=final_dictionary,
        coherence="u_mass"
    ).get_coherence())

Vocabulary Dictionary Size = 580
C_v = 0.5695136863872025
C_umass = -2.5768553446864186


In [43]:
'''
lambda = 0 -> sorts words purely based on exclusivity ==> p(word | topic) / p(word)
lambda = 1 -> sorts words purely based on probability/frequency  within topic ==> p(word | topic)
'''
final_vis = pyLDAvis.gensim.prepare(final_lda_model, final_corpus, final_dictionary, 
                                    mds='mmds', R=25, lambda_step=0.05)
pyLDAvis.display(final_vis)

In [44]:
pyLDAvis.save_html(final_vis, '../figures/LDA_vis_three_topic.html')

> Topic 1 appears to center on negative customer experiences involving service mistakes, uneven or unsatisfactory haircuts, and feelings of disrespect or frustration—often leading customers to leave early, avoid returning, or warn others not to recommend the salon
- service quality issues (main idea in topic 1)

> Topic 2 appears to describe complaints about wait times, poor customer service quality, or just the overall in-salon process/experience (waiting, being attended to, feeling uncared for, etc.)
- waiting time
- negative staff and service interactions

> Topic 3 appears to capture complaints about unprofessional barber behavior and communication style that made the customer feel uncomfortable, disrespected, and annoyed throughout the appointment
- unprofessional conduct during service, communication, and tone by staff
- physical discomfort or rough handling during service

# Conclusion

Our exploration demonstrates that data assumptions, text representation methods, and model selection heavily impact topic modeling outcomes. This notebook details our complete technical implementation—from data cleaning and feature extraction to hyperparameter tuning, model selection, and interpretation. More importantly, it documents the chronological evolution of our data science strategy to solving the topic modeling problem.

### 1. The Pitfall of Flawed Assumptions
Our initial approach failed because we unknowingly applied a **single-topic-per-document** assumption. This bias dictated our data preparation and led us to use **K-Means (hard clustering)**. Because review documents naturally contain multiple themes, forcing a hard division resulted in poor topic coherence and low human interpretability which was seen in the TF-IDF and SBERT K-Means implementation.

### 2. The Breakthrough: Shifting to Soft Clustering
Struggling with the poor results of K-Means led to a critical realization: our foundational assumptions were fundamentally incompatible with the inherent structure of the text. Working with truly messy data revealed that problem-solving must be strictly bound to realistic data presuppositions. By shifting to a **multi-topic-per-document** assumption, we pivoted to **Latent Dirichlet Allocation (LDA, soft clustering)**.

### 3. The Outcome and Key Takeaway
Adjusting our core presupposition immediately yielded superior results. The LDA implementation achieved significantly higher topic coherence and clear semantic separation. The defining takeaway from this project is that data preparation and model choices are only as effective as the underlying assumptions made about the data.
